In [ ]:
!pip install scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [1]:
import pandas as pd

data = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')
data

,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
0,1565930130,20141104T000000,4,3.25,3760,4675,2.0,0,0,3,...,2740,1020,2007,0,98038,47.3862,-122.048,3280,4033,429900.0
1,3279000420,20150115T000000,3,1.75,1460,7800,1.0,0,0,2,...,1040,420,1979,0,98023,47.3035,-122.382,1310,7865,233000.0
2,194000575,20141014T000000,4,1.00,1340,5800,1.5,0,2,3,...,1340,0,1914,0,98116,47.5658,-122.389,1900,5800,455000.0
3,2115510160,20141208T000000,3,1.75,1440,8050,1.0,0,0,3,...,1440,0,1985,0,98023,47.3187,-122.390,1790,7488,258950.0
4,7522500005,20140815T000000,2,1.50,1780,4750,1.0,0,0,4,...,1080,700,1947,0,98117,47.6859,-122.395,1690,5962,555000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,5272200045,20141113T000000,3,1.50,1000,6914,1.0,0,0,3,...,1000,0,1947,0,98125,47.7144,-122.319,1000,6947,378000.0
19996,9578500790,20141111T000000,3,2.50,3087,5002,2.0,0,0,3,...,3087,0,2014,0,98023,47.2974,-122.349,2927,5183,399950.0
19997,7202350480,20140930T000000,3,2.50,2120,4780,2.0,0,0,3,...,2120,0,2004,0,98053,47.6810,-122.032,1690,2650,575000.0
19998,1723049033,20140620T000000,1,0.75,380,15000,1.0,0,0,3,...,380,0,1963,0,98168,47.4810,-122.323,1170,15000,245000.0


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from geopy.distance import great_circle
import xgboost as xgb
import pickle

# Drop zipcode column
data = data.drop(columns=['zipcode'])

# Split into features and target
X = data.drop(columns=['price'])
y = data['price']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define feature engineering functions
def add_numerical_features(df):
    df = df.copy()
    df['log_sqft_living'] = np.log1p(df['sqft_living'])
    df['log_sqft_lot'] = np.log1p(df['sqft_lot'])
    df['log_sqft_living15'] = np.log1p(df['sqft_living15'])
    df['log_sqft_lot15'] = np.log1p(df['sqft_lot15'])
    df['living_to_lot_ratio'] = df['sqft_living'] / df['sqft_lot']
    df['above_to_basement_ratio'] = df['sqft_above'] / (df['sqft_basement'] + 1)
    df['bedrooms_per_sqft'] = df['bedrooms'] / df['sqft_living']
    df['bathrooms_per_sqft'] = df['bathrooms'] / df['sqft_living']
    df['living_to_living15_ratio'] = df['sqft_living'] / df['sqft_living15']
    df['house_age'] = 2025 - df['yr_built']
    df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
    df['years_since_renovation'] = np.where(df['yr_renovated'] > 0, 2025 - df['yr_renovated'], df['house_age'])
    df['sqft_living_squared'] = df['sqft_living'] ** 2
    df['bedrooms_squared'] = df['bedrooms'] ** 2
    return df

def add_interaction_features(df):
    df = df.copy()
    df['bedrooms_bathrooms'] = df['bedrooms'] * df['bathrooms']
    df['sqft_living_grade'] = df['sqft_living'] * df['grade']
    df['age_grade'] = df['house_age'] * df['grade']
    return df

def add_geographical_features(X_train, X_test, kmeans=None):
    X_train = X_train.copy()
    X_test = X_test.copy()
    seattle_center = (47.6062, -122.3321)
    X_train['distance_to_seattle_center'] = X_train.apply(
        lambda row: great_circle((row['lat'], row['long']), seattle_center).km, axis=1)
    X_test['distance_to_seattle_center'] = X_test.apply(
        lambda row: great_circle((row['lat'], row['long']), seattle_center).km, axis=1)

    if kmeans is None:
        kmeans = KMeans(n_clusters=10, random_state=42)
        X_train['geo_cluster'] = kmeans.fit_predict(X_train[['lat', 'long']])
    else:
        X_train['geo_cluster'] = kmeans.predict(X_train[['lat', 'long']])
    X_test['geo_cluster'] = kmeans.predict(X_test[['lat', 'long']])

    return X_train, X_test, kmeans

def encode_categorical(X_train, X_test, encoder=None):
    cat_columns = ['waterfront', 'view']
    if encoder is None:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_train = encoder.fit_transform(X_train[cat_columns])
    else:
        encoded_train = encoder.transform(X_train[cat_columns])

    encoded_test = encoder.transform(X_test[cat_columns])
    encoded_columns = encoder.get_feature_names_out(cat_columns)

    encoded_train_df = pd.DataFrame(encoded_train, columns=encoded_columns, index=X_train.index)
    encoded_test_df = pd.DataFrame(encoded_test, columns=encoded_columns, index=X_test.index)

    X_train = X_train.drop(columns=cat_columns).join(encoded_train_df)
    X_test = X_test.drop(columns=cat_columns).join(encoded_test_df)

    return X_train, X_test, encoder

# Apply feature engineering
X_train = add_numerical_features(X_train)
X_test = add_numerical_features(X_test)
X_train = add_interaction_features(X_train)
X_test = add_interaction_features(X_test)
X_train, X_test, kmeans = add_geographical_features(X_train, X_test)
X_train, X_test, encoder = encode_categorical(X_train, X_test)

# Drop irrelevant or redundant features
drop_columns = ['id', 'date', 'sqft_above', 'sqft_basement']
X_train = X_train.drop(columns=drop_columns, errors='ignore')
X_test = X_test.drop(columns=drop_columns, errors='ignore')

# Train XGBoost model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Evaluate on test set
from sklearn.metrics import mean_squared_error, r2_score
y_pred = xgb_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²: {r2:.2f}")

# Save model and preprocessing objects
with open('xgb_model_no_zipcode.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
with open('onehot_encoder_no_zipcode.pkl', 'wb') as f:
    pickle.dump(encoder, f)
with open('kmeans_no_zipcode.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

Test RMSE: 136743.61
Test R²: 0.88


In [4]:
new_data = pd.read_csv("housing_holdout_test.csv")

new_data = add_numerical_features(new_data)
new_data = add_interaction_features(new_data)

new_data, _, _ = add_geographical_features(new_data, new_data, kmeans=kmeans)

# Apply categorical encoding
new_data, _, _ = encode_categorical(new_data, new_data, encoder=encoder)

# Drop irrelevant or redundant features
new_data = new_data.drop(columns=drop_columns, errors='ignore')

# Ensure columns match training data
new_data = new_data[X_train.columns]

# Make predictions
predictions = xgb_model.predict(new_data)

new_data['predicted_price'] = predictions
new_data.to_csv("new_housing_with_predictions.csv", index=False)
print("Predictions saved to 'new_housing_with_predictions.csv'")

Predictions saved to 'new_housing_with_predictions.csv'
